In [5]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
# 导入随机森林回归器（RF）
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\RF预测结果_RMSE评分.xlsx"  # 文件名还原为RF
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"

df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)

ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:]  # 保留DataFrame格式，用于获取特征名（不直接转np.array）
Y = df_ev.iloc[:, 1:].values

# ==============================================================================
# 3: 数据拆分与建模（RF模型 + 指定参数）
# ==============================================================================
# 数据拆分（X需转为np.array用于建模，特征名从原X获取）
X_arr = X.values  # 新增：将X转为数组用于模型训练
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X_arr, Y, indices,  # 建模使用转换后的X_arr
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels
)

# 定义RF固定参数（移除estimator__前缀，适配MultiOutputRegressor直接实例化）
fixed_params = {
    'max_depth': 20,
    'max_features': 1.0,  # 严格按要求设置为1.0（使用所有特征）
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 100,
    'random_state': 42  # 保留随机种子保证结果可复现
}

# 构建最终模型（先实例化单输出RF模型，拟合后再封装MultiOutputRegressor，避免estimator未拟合）
# 第一步：实例化并拟合单输出RF模型（用于提取特征重要性）
rf_reg = RandomForestRegressor(**fixed_params)
rf_reg.fit(X_tr, Y_tr[:, 0])  # 取Y_tr第一列拟合，与MultiOutputRegressor拟合逻辑一致
# 第二步：封装为多输出模型，复用已拟合的单输出模型参数
model_final = MultiOutputRegressor(rf_reg)
model_final.fit(X_tr, Y_tr)  # 多输出模型拟合，确保整体模型可用

# ==============================================================================
# 新增：特征重要性分析（按用户提供代码功能实现，修复NotFittedError）
# ==============================================================================
# 打印特征排序标题（修正原代码语法错误，改为print函数）
print('特征排序：')
# 获取特征名称（从原X DataFrame获取）
feature_names = X.columns
# 获取特征重要性（使用已拟合的单输出RF模型，避免未拟合报错）
feature_importances = rf_reg.feature_importances_
# 对特征重要性进行排序，获取排序后的索引
indices = np.argsort(feature_importances)
# 打印排序后的索引
print(indices)
# 遍历索引，打印每个特征的名称及对应重要性
for index in indices:
    print('feature %s (%f)' %(feature_names[index], feature_importances[index]))

特征排序：
[ 4 10  9  5  2  7  6  3 11  8  0  1]
feature F5 (0.037159)
feature FZ (0.038618)
feature FXL (0.048502)
feature F6 (0.057197)
feature F3 (0.063761)
feature F8 (0.067959)
feature F7 (0.069619)
feature F4 (0.072563)
feature FY (0.085995)
feature VIS (0.120932)
feature F1 (0.168174)
feature F2 (0.169521)


In [9]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\GBR预测结果_RMSE评分.xlsx"
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"

df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)

ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:]  # 保留DataFrame格式，用于获取特征名（不直接转np.array）
Y = df_ev.iloc[:, 1:].values

# ==============================================================================
# 3: 数据拆分与建模（固定参数）
# ==============================================================================
# 数据拆分（X需转为np.array用于建模，特征名从原X获取）
X_arr = X.values  # 新增：将X转为数组用于模型训练
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X_arr, Y, indices,  # 建模使用转换后的X_arr
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels
)

# 定义固定参数
fixed_params = {
    'learning_rate': 0.08,
    'loss': 'squared_error',
    'max_depth': 4,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200,
    'subsample': 1.0,
    'random_state': 42
}

# 构建最终模型（先实例化单输出GBR模型，拟合后再封装MultiOutputRegressor，避免estimator未拟合）
# 第一步：实例化并拟合单输出GBR模型（用于提取特征重要性）
gbr_reg = GradientBoostingRegressor(**fixed_params)
gbr_reg.fit(X_tr, Y_tr[:, 0])  # 取Y_tr第一列拟合，与MultiOutputRegressor拟合逻辑一致
# 第二步：封装为多输出模型，复用已拟合的单输出模型参数
model_final = MultiOutputRegressor(gbr_reg)
model_final.fit(X_tr, Y_tr)  # 多输出模型拟合，确保整体模型可用

# ==============================================================================
# 新增：特征重要性分析（与RF模型功能一致，修复NotFittedError）
# ==============================================================================
# 打印特征排序标题
print('特征排序：')
# 获取特征名称（从原X DataFrame获取）
feature_names = X.columns
# 获取特征重要性（使用已拟合的单输出GBR模型，避免未拟合报错）
feature_importances = gbr_reg.feature_importances_
# 对特征重要性进行排序，获取排序后的索引
indices = np.argsort(feature_importances)
# 打印排序后的索引
print(indices)
# 遍历索引，打印每个特征的名称及对应重要性
for index in indices:
    print('feature %s (%f)' %(feature_names[index], feature_importances[index]))

特征排序：
[ 3  4  6  5  7 11  2 10  1  0  8  9]
feature F4 (0.006526)
feature F5 (0.012367)
feature F7 (0.015906)
feature F6 (0.015957)
feature F8 (0.028891)
feature FY (0.046384)
feature F3 (0.055629)
feature FZ (0.065669)
feature F2 (0.161878)
feature F1 (0.162131)
feature VIS (0.206428)
feature FXL (0.222237)


In [12]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
# 替换为AdaBoost回归器
from sklearn.ensemble import AdaBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\AdaBoost预测结果_RMSE评分.xlsx"  # 文件名同步更新
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"

df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)

ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:]  # 保留DataFrame格式，用于获取特征名（不直接转np.array）
Y = df_ev.iloc[:, 1:].values

# ==============================================================================
# 3: 数据拆分与建模（更新为AdaBoost参数）
# ==============================================================================
# 数据拆分（X需转为np.array用于建模，特征名从原X获取）
X_arr = X.values  # 新增：将X转为数组用于模型训练
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X_arr, Y, indices,  # 建模使用转换后的X_arr
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels
)

# 定义AdaBoost固定参数（移除estimator__前缀，适配MultiOutputRegressor）
fixed_params = {
    'learning_rate': 0.1,
    'loss': 'linear',
    'n_estimators': 50,
    'random_state': 42  # 保留随机种子保证结果可复现
}

# 构建最终模型（先实例化单输出AdaBoost模型，拟合后再封装MultiOutputRegressor，避免estimator未拟合）
# 第一步：实例化并拟合单输出AdaBoost模型（用于提取特征重要性）
ada_reg = AdaBoostRegressor(**fixed_params)
ada_reg.fit(X_tr, Y_tr[:, 0])  # 取Y_tr第一列拟合，与MultiOutputRegressor拟合逻辑一致
# 第二步：封装为多输出模型，复用已拟合的单输出模型参数
model_final = MultiOutputRegressor(ada_reg)
model_final.fit(X_tr, Y_tr)  # 多输出模型拟合，确保整体模型可用

# ==============================================================================
# 新增：特征重要性分析（与RF、GBR模型功能一致，修复NotFittedError）
# ==============================================================================
# 打印特征排序标题
print('特征排序：')
# 获取特征名称（从原X DataFrame获取）
feature_names = X.columns
# 获取特征重要性（使用已拟合的单输出AdaBoost模型，避免未拟合报错）
feature_importances = ada_reg.feature_importances_
# 对特征重要性进行排序，获取排序后的索引
indices = np.argsort(feature_importances)
# 打印排序后的索引
print(indices)
# 遍历索引，打印每个特征的名称及对应重要性
for index in indices:
    print('feature %s (%f)' %(feature_names[index], feature_importances[index]))

特征排序：
[ 4  7  5 11 10  2  0  9  6  1  8  3]
feature F5 (0.011200)
feature F8 (0.022431)
feature F6 (0.023111)
feature FY (0.036984)
feature FZ (0.037453)
feature F3 (0.039939)
feature F1 (0.112769)
feature FXL (0.118912)
feature F7 (0.133688)
feature F2 (0.138339)
feature VIS (0.145612)
feature F4 (0.179562)


In [14]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
# 导入极端随机树回归器（ETR）
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\ETR预测结果_RMSE评分.xlsx"  # 文件名同步更新为ETR
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"

df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)

ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:]  # 保留DataFrame格式，用于获取特征名（不直接转np.array）
Y = df_ev.iloc[:, 1:].values

# ==============================================================================
# 3: 数据拆分与建模（ETR模型 + 指定参数）
# ==============================================================================
# 数据拆分（X需转为np.array用于建模，特征名从原X获取）
X_arr = X.values  # 新增：将X转为数组用于模型训练
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X_arr, Y, indices,  # 建模使用转换后的X_arr
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels
)

# 定义ETR固定参数（移除estimator__前缀，适配MultiOutputRegressor）
fixed_params = {
    'max_depth': 15,
    'max_features': 'sqrt',  # 严格按要求设置为平方根特征数
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200,
    'random_state': 42  # 保留随机种子保证结果可复现
}

# 构建最终模型（先实例化单输出ETR模型，拟合后再封装MultiOutputRegressor，避免estimator未拟合）
# 第一步：实例化并拟合单输出ETR模型（用于提取特征重要性）
etr_reg = ExtraTreesRegressor(**fixed_params)
etr_reg.fit(X_tr, Y_tr[:, 0])  # 取Y_tr第一列拟合，与MultiOutputRegressor拟合逻辑一致
# 第二步：封装为多输出模型，复用已拟合的单输出模型参数
model_final = MultiOutputRegressor(etr_reg)
model_final.fit(X_tr, Y_tr)  # 多输出模型拟合，确保整体模型可用

# ==============================================================================
# 新增：特征重要性分析（与RF、GBR、AdaBoost模型功能一致，修复NotFittedError）
# ==============================================================================
# 打印特征排序标题
print('特征排序：')
# 获取特征名称（从原X DataFrame获取）
feature_names = X.columns
# 获取特征重要性（使用已拟合的单输出ETR模型，避免未拟合报错）
feature_importances = etr_reg.feature_importances_
# 对特征重要性进行排序，获取排序后的索引
indices = np.argsort(feature_importances)
# 打印排序后的索引
print(indices)
# 遍历索引，打印每个特征的名称及对应重要性
for index in indices:
    print('feature %s (%f)' %(feature_names[index], feature_importances[index]))

特征排序：
[ 2  4  3  1 10  7  0  9 11  5  8  6]
feature F3 (0.049713)
feature F5 (0.051022)
feature F4 (0.053399)
feature F2 (0.057740)
feature FZ (0.062109)
feature F8 (0.063827)
feature F1 (0.078035)
feature FXL (0.107741)
feature FY (0.108892)
feature F6 (0.109552)
feature VIS (0.123005)
feature F7 (0.134964)
